# Quarterly Skills-Related Underemployment (SRU) by Age Group

## Dataset Description
Number and proportion of working individuals with tertiary education who are employed in **semi-skilled or low-skilled jobs**, broken down by **age group**. Quarterly data from DOSM's Labour Force Survey.

> **Why this dataset instead of Salaries by Industry?**
> The DOSM Salaries & Wages Survey (industry × sex) was hosted on `storage.googleapis.com/dosm-public-economy` — an old DOSM Google Cloud bucket that is now inaccessible from Malaysian networks. No equivalent exists on `storage.dosm.gov.my`. This SRU dataset is a direct and arguably better replacement for EduNilai, as it directly measures graduate underemployment rather than just average industry wages.

## Data Source
| Field | Detail |
|---|---|
| **Publisher** | Department of Statistics Malaysia (DOSM) |
| **Survey** | Labour Force Survey (LFS), quarterly |
| **Catalogue** | https://open.dosm.gov.my/data-catalogue/lfs_qtr_sru_age |
| **Parquet URL** | `https://storage.dosm.gov.my/labour/lfs_qtr_sru_age.parquet` |
| **CSV URL** | `https://storage.dosm.gov.my/labour/lfs_qtr_sru_age.csv` |
| **License** | CC BY 4.0 — DOSM |
| **Coverage** | 2016 Q1 – 2025 Q3 |

## Column Descriptions
| Column | Type | Description |
|---|---|---|
| `date` | date | Quarter start date (YYYY-MM-DD, DD always = 01) |
| `variable` | string | `"persons"` = count ('000); `"rate"` = proportion (%) |
| `age` | string | Age group — e.g. `15-24`, `25-34`, `35-44`, `45-54`, `55-64` |
| `sru` | float | Value — either headcount ('000) or rate (%) depending on `variable` |

> **Long format:** Filter `variable == 'rate'` for the underemployment rate (%), or `variable == 'persons'` for the headcount in thousands.

## Relevance to EduNilai
Age breakdown reveals whether **young fresh graduates (15–34)** face higher underemployment than older workers — critical for estimating the payback period of a degree. If young graduates are disproportionately underemployed in their early career years, the ROI calculation must account for a longer period of sub-optimal earnings.

In [ ]:
# pip install pandas fastparquet
import pandas as pd
import matplotlib.pyplot as plt
import os

URL = 'https://storage.dosm.gov.my/labour/lfs_qtr_sru_age.parquet'
df = pd.read_parquet(URL)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df.head(12)

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nDate range:", df['date'].min(), "–", df['date'].max())
print("\nUnique variable:", df['variable'].unique())
print("Unique age groups:", sorted(df['age'].unique()))

In [ ]:
# Split long format into rate and persons
df_rate    = df[df['variable'] == 'rate'].copy()
df_persons = df[df['variable'] == 'persons'].copy()

print("Latest quarter — SRU rate by age group:")
latest = df_rate[df_rate['date'] == df_rate['date'].max()]
print(latest[['age', 'sru']].sort_values('age').to_string(index=False))

In [ ]:
# Plot: SRU rate by age group over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for age, grp in df_rate.groupby('age'):
    axes[0].plot(grp['date'], grp['sru'], label=age, marker='o', markersize=3)
axes[0].set_title('Graduate Underemployment Rate by Age Group (%)')
axes[0].set_ylabel('SRU Rate (%)')
axes[0].set_xlabel('Quarter')
axes[0].legend(fontsize=8)

# Latest quarter bar chart
latest_sorted = latest.sort_values('age')
axes[1].bar(latest_sorted['age'], latest_sorted['sru'], color='steelblue')
axes[1].set_title(f"SRU Rate by Age — {latest['date'].max().strftime('%Y Q') + str((latest['date'].max().month-1)//3+1)}")
axes[1].set_ylabel('SRU Rate (%)')
axes[1].set_xlabel('Age Group')

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../data/salary', exist_ok=True)
df.to_csv('../data/salary/graduate_underemployment_age.csv', index=False, encoding='utf-8')
print("Saved -> ../data/salary/graduate_underemployment_age.csv")
print(f"Rows: {len(df)}")
print("\nTip: filter by variable column before use:")
print("  df[df['variable'] == 'rate']    → underemployment rate (%)")
print("  df[df['variable'] == 'persons'] → headcount in thousands")